Step 1 — Load

In [ ]:
import pandas as pd


Step 2 — Sanity checks

In [ ]:
ad = pd.read_csv("AD_zscore.csv", index_col=0)
pd_ds = pd.read_csv("PD_zscore.csv", index_col=0)
hd = pd.read_csv("HD_zscore.csv", index_col=0)
ms = pd.read_csv("MS_zscore.csv", index_col=0)

metadata = pd.read_csv("sample_metadata_all_diseases.csv")


In [ ]:
assert list(ad.index) == list(pd_ds.index) == list(hd.index) == list(ms.index)
print("✅ Gene order identical across datasets")


✅ Gene order identical across datasets


In [ ]:
all_samples = (
    list(ad.columns) +
    list(pd_ds.columns) +
    list(hd.columns) +
    list(ms.columns)
)

missing = set(all_samples) - set(metadata["sample_id"])
print("Missing samples in metadata:", len(missing))


Missing samples in metadata: 0


Step 3 — Combine expression matrices

In [ ]:
X = pd.concat(
    [ad, pd_ds, hd, ms],
    axis=1
)


In [ ]:
X.shape


(14784, 287)

Step 4 — Create label vector y

In [ ]:
metadata = metadata.set_index("sample_id")
y = metadata.loc[X.columns, "label"]


In [ ]:
y.value_counts()


,count
label,
0,212
1,33
3,20
4,14
2,8


Step 5 — Transpose for ML

In [ ]:
X_ml = X.T


In [ ]:
print("X_ml shape:", X_ml.shape)
print("y shape:", y.shape)


X_ml shape: (287, 14784)
y shape: (287,)


In [ ]:
X_ml.to_csv("X_multidisease_zscore.csv")
y.to_csv("y_multidisease_labels.csv")


Step 6: Final Dataset Split (Train / Validation / Test)


In [ ]:
from sklearn.model_selection import train_test_split

# First split: Train vs Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_ml, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Second split: Validation vs Test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Train: (200, 14784) (200,)
Val:   (43, 14784) (43,)
Test:  (44, 14784) (44,)


Step 7: Model-Ready Data Export


In [ ]:
import pandas as pd

pd.DataFrame(X_train).to_csv("X_train.csv", index=False)
pd.DataFrame(X_val).to_csv("X_val.csv", index=False)
pd.DataFrame(X_test).to_csv("X_test.csv", index=False)

pd.Series(y_train).to_csv("y_train.csv", index=False)
pd.Series(y_val).to_csv("y_val.csv", index=False)
pd.Series(y_test).to_csv("y_test.csv", index=False)


Step 8: Class Distribution Check


In [ ]:
import pandas as pd

label_counts = pd.Series(y_train).value_counts().sort_index()
label_counts


,count
label,
0,148
1,23
2,5
3,14
4,10


A harmonized multi-disease transcriptomic dataset comprising 14,784 common genes across 287 samples was constructed. After normalization, gene symbol harmonization, duplicate removal, and z-score standardization, the dataset was split into training, validation, and test sets for downstream AI modeling.